In [3]:
# Installation des dépendances
!pip install scenedetect ultralytics open_clip_torch

In [4]:
import os, cv2, torch, numpy as np, pandas as pd
from tqdm import tqdm
from scenedetect import SceneManager, ContentDetector, VideoStreamCv2
from ultralytics import YOLO
import open_clip
from PIL import Image
from google.colab import drive

# 1. Montage du Drive
drive.mount('/content/drive')

# 2. Configuration des modèles (GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
yolo_model = YOLO('yolov8n.pt').to(device)
clip_model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
clip_model = clip_model.to(device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



In [18]:
def process_video_full(video_path):
    # On initialise les variables pour éviter des erreurs de référence
    cap = None
    video_stream = None

    try:

        # --- 2. SÉMANTIQUE & MOUVEMENT - Lecture avec OpenCV ---
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return None

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        indices = [int(total_frames * 0.2), int(total_frames * 0.5), int(total_frames * 0.8)]
        frames_for_clip = []
        yolo_counts = {'person': 0, 'skis': 0, 'snowboard': 0}


        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret or frame is None: continue

            # YOLO
            y_res = yolo_model(frame, imgsz=320, verbose=False, conf=0.25)[0]
            for c in y_res.boxes.cls:
                label = y_res.names[int(c)]
                if label in yolo_counts: yolo_counts[label] += 1


            # CLIP
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames_for_clip.append(preprocess(Image.fromarray(frame_rgb)).unsqueeze(0))

        # --- 3. CLIP BATCH ---
        avg_embedding = np.zeros(512)
        if frames_for_clip:
            img_input = torch.cat(frames_for_clip).to(device)
            with torch.no_grad():
                emb = clip_model.encode_image(img_input).mean(dim=0).cpu().numpy()
                avg_embedding = emb


        # --- 4. RÉSULTATS ---
        res = {
            'video_id': os.path.splitext(os.path.basename(video_path))[0],
            'yolo_person': yolo_counts['person'],
            'yolo_ski_snow': yolo_counts['skis'] + yolo_counts['snowboard']
        }
        for i, v in enumerate(avg_embedding):
            res[f'c{i}'] = round(float(v), 4)

        return res

    except Exception as e:
        # Nettoyage en cas d'erreur
        print(f"Erreur sur {os.path.basename(video_path)}: {e}")
        return None

In [22]:
# --- BOUCLE PRINCIPALE ---
VIDEO_DIR = "/content/drive/MyDrive/hackathon/videos/" # <--- MODIFIE CE CHEMIN
paths = [os.path.join(r, f) for r, _, fs in os.walk(VIDEO_DIR) for f in fs if f.lower().endswith(('.mp4', '.mov'))]

results = []

for p in tqdm(paths):
  data = process_video_full(p)
  if data: results.append(data)

# Sauvegarde sur le Drive
df = pd.DataFrame(results)
df.to_csv("/content/drive/MyDrive/hackathon/features_semantique.csv", index=False)

100%|██████████| 1686/1686 [56:38<00:00,  2.02s/it]


In [21]:
df.head()

,video_id,yolo_person,yolo_ski_snow,c0,c1,c2,c3,c4,c5,c6,...,c502,c503,c504,c505,c506,c507,c508,c509,c510,c511
0,VIDEO_7091569180054490373,2,0,-0.0642,1.4171,-0.1620,0.2403,-0.2569,0.1202,0.1216,...,0.3261,-0.6892,-0.1823,-0.1186,0.4953,0.1719,0.2627,0.0184,0.1539,-0.2005
1,VIDEO_7090931568486763781,12,5,-0.4177,0.1856,0.0503,0.8689,0.3162,0.5331,-0.0466,...,-0.3247,-1.0833,0.3180,0.0358,0.3494,0.0625,-0.0527,-0.2539,0.1902,-0.0271
2,VIDEO_7093804812864670981,3,0,-0.4251,0.1215,1.1319,-0.1833,-0.0030,0.0973,0.3494,...,0.0236,-1.0790,-0.4598,-0.2391,0.0770,-0.2096,0.1614,-0.1438,-0.1508,0.5514
3,VIDEO_7088619536202648837,0,0,-0.2347,1.4549,0.4907,0.1776,0.0663,0.3282,0.0566,...,0.1370,-0.0534,0.1173,-0.2281,-0.1629,0.2089,-0.0216,-0.1347,-0.1230,0.1084
4,VIDEO_7087984641864502534,4,0,-0.5271,-0.4666,0.5909,0.5538,-0.1423,-0.0248,0.6020,...,-0.4524,-1.0225,0.2844,0.4425,0.3393,-0.6340,0.1002,-0.7794,0.3267,-0.0264
